# Data Analysis

## Preparations

First of all, we read the raw results.

In [ ]:
import pandas as pd

import os

from pathlib import Path


ROOT = Path(os.getcwd())


def data_dir(root: Path) -> Path:
    return root / "data"


# for the export of tables and plots
def output(root: Path) -> Path:
    return data_dir(root) / "output"


def raw_results(root: Path) -> Path:
    return data_dir(root) / "raw_results" / "raw_results.csv"


# for the export of LaTeX tables:
def latex_tables(root: Path) -> Path:
    return output(root) / "latex_tables"


# for the export of figures:
def figures(root: Path) -> Path:
    return output(root) / "figures"


rr = pd.read_csv(
    raw_results(ROOT),
    low_memory=False,
)

Then, we drop columns that are actually not usable.

In [ ]:
COLS_OF_INTEREST = [
    "full_name_of_repo",
    "commit_sha",
    "path",
    "is_ccdc_event",
    "detected_channel",
    "created_at",
    "pushed_at",
    "updated_at",
    "date",
]

UNNECESSARY_COLS = []

for col in rr.columns:
    if col not in COLS_OF_INTEREST:
        UNNECESSARY_COLS.append(col)

rr.drop(columns=UNNECESSARY_COLS, inplace=True)

Next, we perform data conversion.

In [ ]:
def to_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    s = series.astype("string").str.strip().str.lower()
    mapping = {"true": True, "false": False}
    return s.map(mapping)


rr["is_ccdc_event"] = to_bool(rr["is_ccdc_event"])

rr["detected_channel"] = rr["detected_channel"].astype("string").fillna("")

DATETIME_COLS = ["created_at", "pushed_at", "updated_at", "date"]

for col in DATETIME_COLS:
    if col in rr.columns:
        rr[col] = pd.to_datetime(rr[col], errors="coerce", utc=True)


At this point, our results look like this:

In [ ]:
rr.head()

From here on, we work with a new Python variable/identifier.

In [ ]:
results = rr.copy()

Now, we remove the channels detected for the subjects that are **not** CCDC events.

In [ ]:
results["detected_channel"] = (
    results["detected_channel"]
        .where(results["is_ccdc_event"] == True, "")
)

Next, we transform the data frame: each row should be a unique subject – one row per subject.

In [ ]:
KEY_COLS = [
    "full_name_of_repo",
    "commit_sha",
    "path",
]

gb = results.groupby(KEY_COLS, dropna=False)


def agg_channels(x: pd.Series) -> tuple[str, ...]:
    vals = [
        v
        for v in x.astype("string").tolist()
        if isinstance(v, str) and v.strip() != ""
    ]
    return tuple(sorted(set(vals)))


agg = gb.agg(
    is_ccdc_event=("is_ccdc_event", "first"),
    detected_channels=("detected_channel", agg_channels),
)

for col in COLS_OF_INTEREST:
    if col not in KEY_COLS and col not in agg.columns and col != "detected_channel":
        agg[col] = gb[col].first()

results = agg.reset_index()

At this point, we start extracting further information based on the metadata already present in the data frame.

We define the first activity of a repository to be the first point in time providing evidence for project activity: either the time the repository was created on GitHub or the time of the first commit.

In [ ]:
results["first_activity"] = (
    results[["date", "created_at"]]
        .min(axis=1)
        .groupby(results["full_name_of_repo"], dropna=False)
        .transform("min")
)

The repository age is the simply the time passed since the repository's first activity.

In [ ]:
import numpy as np

results["repo_age"] = (
    (results["date"] - results["first_activity"]).to_numpy()
    / np.timedelta64(1, "D")
)

results.sort_values(["full_name_of_repo", "repo_age"], inplace=True)
results.reset_index(drop=True, inplace=True)

Based on the age of the repository at the time of a commit, we can assign the subjects to a set of age groups.

In [ ]:
AGE_GROUPS = [
    (0, 1, "0-1"),
    (1, 2, "1-2"),
    (2, 3, "2-3"),
    (3, 4, "3-4"),
    (4, 5, "4-5"),
    (5, 6, "5-6"),
    (6, 7, "6-7"),
    (7, 8, "7-8"),
    (8, 9, "8-9"),
    (9, 10, "9-10"),
    (10, 11, "10-11"),
    (11, 12, "11-12"),
    (12, 13, "12-13"),
    (13, 14, "13-14"),
    (14, 15, "14-15"),
    (15, 999, "15+"),
]


def assign_age_group(age_in_days: float) -> int | None:
    if pd.isna(age_in_days):
        return None
    for lo, hi, label in AGE_GROUPS:
        if lo * 365.25 <= age_in_days < hi * 365.25:
            return lo
    return None


results["age_group"] = results["repo_age"].apply(assign_age_group)

Later, we will group subjects by the year of their commits. Out of convenience, we introduce an extra column for that.

In [ ]:
results["year"] = results["date"].dt.year

Now, we create a dedicated data frame for the repository demographics.

In [ ]:
repos = (
    results
        .groupby("full_name_of_repo", as_index=False)
        .agg(
            created_at=("created_at", "first"),
            pushed_at=("pushed_at", "first"),
            updated_at=("updated_at", "first"),
            first_activity=("first_activity", "first"),
            first_subject_at=("date", "min"),
            last_subject_at=("date", "max"),
        )
)

This allows us to remove repository metadata from the results data frame.

In [ ]:
results.drop(
    columns=["created_at", "pushed_at", "updated_at", "first_activity"],
    inplace=True,
)

Since we introduced *first_activity*, consequently we introduce *last_activity*: the timestamp that is more recent, *pushed_at* or *updated_at*.

In [ ]:
repos["last_activity"] = repos[["pushed_at", "updated_at"]].max(axis=1)

Then we create columns for the age (in days), the age group, and the age group at the time of a repository's first subject.

In [ ]:

repos["age"] = (
    (repos["last_activity"] - repos["first_activity"]).to_numpy()
    / np.timedelta64(1, "D")
)

repos["age_group"] = repos["age"].apply(assign_age_group)

repos["days_until_first_subject"] = (
    (repos["first_subject_at"] - repos["first_activity"]).to_numpy()
    / np.timedelta64(1, "D")
)
repos["age_group_at_first_subject"] = repos["days_until_first_subject"].apply(assign_age_group)

repos.drop(columns=["days_until_first_subject"], inplace=True)
repos.sort_values("full_name_of_repo", ascending=False, inplace=True)
repos.reset_index(drop=True, inplace=True)

Our repository data frame looks like this:

In [ ]:
repos.head()

The base sample of repositories is from June 2019. Hence, we remove any subjects originating from repositories that were added to GitHub after June 2019.

In [ ]:
repos = repos.set_index("full_name_of_repo")
results = results.set_index("full_name_of_repo")

mask = repos["created_at"] >= "2019-07-01"

repos = repos.loc[~mask]
results = results.loc[results.index.isin(repos.index)]
repos.reset_index(inplace=True)
results.reset_index(inplace=True)

We make the path values case insensitive to only analyze how README and CONTRIBUTING files are distributed over time independent of a project maintainers opinion on whether or not a README/CONTRIBUTING file should be named in ALL CAPS.

In [ ]:
results["path"] = (
    results["path"]
    .str.split(".", n=1)
    .apply(lambda x: x[0].upper() + ("." + x[1] if len(x) > 1 else ""))
)

Since I do not know whether and how there are differences between CONTRIBUTING files and README files regarding what project maintainers decide to document, I decided to define a dedicated data frame for README files only. This allows for a more nuanced analysis later.

**However, originally I had included CONTRIBUTING files arguing that more and more projects these days point to their CONTRIBUTING files for communication channel relevant information. So, it makes sense, to keep the subjects that come from CONTRIBUTING files in the data set of the analysis.**

In [ ]:
# readme_results = results[results["path"].str.contains("README", case=False, na=False)]

We split the results/subjects in two: subjects before the base sample was drawn and subjects after June 2019. Why? Because if it was true that projects create CCDC events in their early years, not having any new projects for the time after June 2019 would create a bias in the data. How would we be able to analyze trends over time?

In [ ]:
results_before_july_2019 = results[results["date"] < "2019-07-01"]
results_since_july_2019 = results[results["date"] >= "2019-07-01"]

## Setup and Helper Functions

In [ ]:
from collections.abc import Callable, Sequence
from dataclasses import dataclass


MetricFn = Callable[[pd.DataFrame], object]


def _require_cols(g: pd.DataFrame, cols: Sequence[str]) -> None:
    missing = [c for c in cols if c not in g.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------
def m_n_subjects(g: pd.DataFrame) -> int:
    return int(len(g))


def m_n_repos(g: pd.DataFrame) -> int:
    _require_cols(g, ["full_name_of_repo"])
    return int(g["full_name_of_repo"].nunique())


def m_n_commits(g: pd.DataFrame) -> int:
    _require_cols(g, ["commit_sha"])
    return int(g["commit_sha"].nunique())


def m_n_distinct_paths(g: pd.DataFrame) -> int:
    _require_cols(g, ["path"])
    return int(g["path"].nunique())


def m_positive_rate(g: pd.DataFrame) -> float:
    _require_cols(g, ["is_ccdc_event"])
    n = len(g)
    return np.nan if n == 0 else float(g["is_ccdc_event"].sum() / n)


def m_n_distinct_channels(g: pd.DataFrame) -> int:
    _require_cols(g, ["detected_channels"])
    return int(
        g["detected_channels"]
        .explode()
        .dropna()
        .nunique()
    )


# ---------------------------------------------------------------------
# Registry: metric_name -> (fn, required_columns)
# ---------------------------------------------------------------------
METRICS: dict[str, tuple[MetricFn, tuple[str, ...]]] = {
    "n_subjects": (m_n_subjects, ()),
    "n_repos": (m_n_repos, ("full_name_of_repo",)),
    "n_commits": (m_n_commits, ("commit_sha",)),
    "n_distinct_paths": (m_n_distinct_paths, ("path",)),
    "positive_rate": (m_positive_rate, ("is_ccdc_event",)),
    "n_distinct_channels": (
        m_n_distinct_channels,
        ("detected_channels",),
    ),
}

ALL_METRICS = (
    "n_subjects",
    "n_repos",
    "n_commits",
    "n_distinct_paths",
    "positive_rate",
    "n_distinct_channels",
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SummarizeConfig:
    include_metrics: tuple[str, ...]
    group_keys: tuple[str, ...] = ()
    forbid_access_to_group_keys: bool = True
    drop_group_keys_from_frame: bool = True
    deny_columns: tuple[str, ...] = ()


# ---------------------------------------------------------------------
# Core summarization
# ---------------------------------------------------------------------
def summarize_subjects_configurable(
    g: pd.DataFrame,
    cfg: SummarizeConfig,
) -> pd.Series:

    g_eff = (
        g.drop(columns=list(cfg.group_keys), errors="ignore")
        if cfg.drop_group_keys_from_frame and cfg.group_keys
        else g
    )

    forbidden = set(cfg.deny_columns)
    if cfg.forbid_access_to_group_keys:
        forbidden |= set(cfg.group_keys)

    out: dict[str, object] = {}

    for name in cfg.include_metrics:
        if name not in METRICS:
            raise KeyError(
                f"Unknown metric: {name}. "
                f"Known: {sorted(METRICS)}"
            )

        fn, required = METRICS[name]

        illegal = [c for c in required if c in forbidden]
        if illegal:
            raise ValueError(
                f"Metric '{name}' requires forbidden columns {illegal}. "
                f"Forbidden: {sorted(forbidden)}"
            )

        out[name] = fn(g_eff)

    return pd.Series(out)

In [ ]:
def distribution_stats(
    s: pd.Series,
    *,
    prefix: str = "",
    dropna: bool = True,
) -> pd.Series:
    """
    Core descriptive stats for a numeric series, including boxplot whiskers/outliers.
    Works for both raw numeric data and 'counts per category' (value_counts output values).
    """
    if dropna:
        s = s.dropna()

    if s.empty:
        return pd.Series(dtype="float64")

    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return pd.Series(dtype="float64")

    q1 = s.quantile(0.25)
    q2 = s.quantile(0.50)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    # Handle iqr=0 robustly (all values equal) without producing empty slices
    if pd.isna(iqr) or iqr == 0:
        lower_whisker = s.min()
        upper_whisker = s.max()
        n_outliers = 0
    else:
        lo = q1 - 1.5 * iqr
        hi = q3 + 1.5 * iqr
        lower_whisker = s[s >= lo].min()
        upper_whisker = s[s <= hi].max()
        n_outliers = int(((s < lower_whisker) | (s > upper_whisker)).sum())

    def k(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    return pd.Series({
        k("n"): int(len(s)),
        k("min"): float(s.min()),
        k("max"): float(s.max()),
        k("mean"): float(s.mean()),
        k("median"): float(q2),
        k("q1"): float(q1),
        k("q3"): float(q3),
        k("iqr"): float(iqr),
        k("lower_whisker"): float(lower_whisker),
        k("upper_whisker"): float(upper_whisker),
        k("n_outliers"): int(n_outliers),
    })


def boxplot_stats(s: pd.Series) -> pd.Series:
    # Now just a thin wrapper around the shared core
    return distribution_stats(s)


def value_counts_stats(vc: pd.Series) -> pd.Series:
    """
    Stats for a value_counts() result: distribution stats of the counts + concentration metrics.
    Expects vc to be sorted descending (as value_counts() returns by default).
    """
    vc = vc.dropna()
    if vc.empty:
        return pd.Series(dtype="float64")

    total = int(vc.sum())
    n_categories = int(len(vc))

    # shared distribution/boxplot stats over "counts per category"
    core = distribution_stats(vc, prefix="count_")

    # concentration / dominance (requires descending order)
    top_1 = int(vc.iloc[0])
    top_5_sum = int(vc.iloc[:5].sum())
    top_10_sum = int(vc.iloc[:10].sum())

    conc = pd.Series({
        "n_categories": n_categories,
        "total_count": total,
        "top_1_count": top_1,
        "top_1_share": (top_1 / total) if total else float("nan"),
        "top_5_share": (top_5_sum / total) if total else float("nan"),
        "top_10_share": (top_10_sum / total) if total else float("nan"),
    })

    return pd.concat([conc, core])

In [ ]:
from collections.abc import Iterable

def col_to_int(df: pd.DataFrame, int_cols: Iterable[str]) -> pd.DataFrame:
    cols = [c for c in int_cols if c in df.columns]
    df[cols] = df[cols].astype("Int64")
    return df


INT_COLS_OF_SUMMARY = [
    "n_subjects",
    "n_repos",
    "n_commits",
    "n_distinct_paths",
    "n_distinct_channels",
]

INT_COLS_OF_VC_STATS = [
    "n_categories",
    "total_count",
    "min_count",
    "max_count",
    "median_count",
    "q1_count",
    "q3_count",
    "iqr_count",
    "top_1_count",
]

In [ ]:
def vc_detected_channels(
    g: pd.DataFrame,
    *,
    drop_empty: bool = True,
) -> pd.Series:
    """
    Returns value_counts of channels for one group g.
    Assumes g["detected_channels"] contains iterables/tuples of channels, possibly empty () or NaN.
    """
    col: str = "detected_channels"
    s = g[col]

    # explode expects list-like; NaN stays NaN; () becomes empty -> drops on explode
    exploded = s.explode()

    # Optional cleanup
    exploded = exploded.dropna()
    exploded = exploded.astype("string")

    if drop_empty:
        exploded = exploded[exploded.str.strip() != ""]

    # value_counts -> counts per channel
    vc = exploded.value_counts(dropna=False)

    # Make the result stable/consistent
    vc.index.name = "channel"
    vc.name = "count"
    return vc

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # --- text / fonts ---
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # --- lines ---
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # --- axes / spines ---
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.3,

    # --- figure / export ---
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def set_size(width_mm=85, height_mm=60):
    w = width_mm / 25.4
    h = height_mm / 25.4
    return (w, h)

### IEEE / LaTeX-Ready Plotting Setup

In [ ]:
# IEEE / LaTeX-ready plotting setup for Jupyter notebooks
# -------------------------------------------------------
# This cell configures Matplotlib (and optionally Seaborn) so that figures
# integrate well into an IEEE paper and can be exported as PDF/PGF/PNG.

from pathlib import Path
import math

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: set this to True only if a LaTeX installation is available
USE_LATEX = False

# IEEE column widths (in pt)
IEEE_SINGLE_COLUMN_PT = 252.0
IEEE_DOUBLE_COLUMN_PT = 516.0

def pt_to_inch(pt: float) -> float:
    return pt / 72.27

def ieee_figsize(
    width: str = "single",
    fraction: float = 1.0,
    ratio: float = (math.sqrt(5) - 1) / 2,
) -> tuple[float, float]:
    """
    Return figure size in inches for IEEE papers.

    Parameters
    ----------
    width : {"single", "double"} or float
        "single"  -> IEEE single-column width
        "double"  -> IEEE double-column width
        float     -> custom width in pt
    fraction : float
        Fraction of the target width to use.
    ratio : float
        Height/width ratio. Default: golden ratio.
    """
    if width == "single":
        width_pt = IEEE_SINGLE_COLUMN_PT
    elif width == "double":
        width_pt = IEEE_DOUBLE_COLUMN_PT
    elif isinstance(width, (int, float)):
        width_pt = float(width)
    else:
        raise ValueError("width must be 'single', 'double', or a numeric pt value.")

    fig_width_in = pt_to_inch(width_pt) * fraction
    fig_height_in = fig_width_in * ratio
    return fig_width_in, fig_height_in

# Global style configuration
# sns.set_theme(style="whitegrid")
sns.set_theme(
    style="whitegrid",
    palette="colorblind"
)

mpl.rcParams.update({
    # Text + fonts
    "text.usetex": USE_LATEX,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 7,
    "legend.title_fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    # Figure + axes
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.4,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.6,
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Ticks
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4,
    "xtick.direction": "out",
    "ytick.direction": "out",

    # Legend
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.fancybox": False,
    "legend.edgecolor": "0.8",

    # Export
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,   # better editable text in vector exports
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# PGF export settings for direct LaTeX integration
if USE_LATEX:
    mpl.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "pgf.rcfonts": False,
        "text.latex.preamble": r"""
            \usepackage[T1]{fontenc}
            \usepackage[utf8]{inputenc}
            \usepackage{amsmath}
            \usepackage{siunitx}
        """,
    })

# Output directory for exported figures
FIG_DIR = figures(ROOT)
FIG_DIR.mkdir(exist_ok=True)

def save_ieee_figure(fig, filename: str, save_pdf: bool = True, save_pgf: bool = True, save_png: bool = True):
    """
    Save a figure in IEEE-/LaTeX-friendly formats.

    Example
    -------
    fig, ax = plt.subplots(figsize=ieee_figsize("single"))
    ...
    save_ieee_figure(fig, "my_plot", save_pdf=True, save_pgf=True)
    """
    if save_pdf:
        fig.savefig(FIG_DIR / f"{filename}.pdf")
    if save_pgf:
        fig.savefig(FIG_DIR / f"{filename}.pgf")
    if save_png:
        fig.savefig(FIG_DIR / f"{filename}.png")

print("IEEE LaTeX plotting setup loaded.")
print(f"Single-column figure size: {ieee_figsize('single')}")
print(f"Double-column figure size: {ieee_figsize('double')}")

# Example usage:
# fig, ax = plt.subplots(figsize=ieee_figsize("single"))
# ax.plot(x, y)
# ax.set_xlabel("Year")
# ax.set_ylabel("Number of Subjects")
# save_ieee_figure(fig, "example_plot", save_pdf=True, save_pgf=True)
# plt.show()

## Demographics

### Subject Count per Year

The following Code cell prepares a data frame called **time_results** that does not include subjects belonging to a year that has less subjects than the first quantile of the *subject count per year*.

In [ ]:
subject_count_per_year = (
    results
    .groupby("year")
    .size()
)

_to_be_removed = subject_count_per_year[subject_count_per_year < int(np.round(subject_count_per_year.quantile(0.25)))]

time_results = results[
    ~results["year"].isin(_to_be_removed.index)
]

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = subject_count_per_year.to_frame("count")

(
    _df
        .rename_axis("Year", axis="index")
        .rename(
            columns={
                "count": r"$n_{\mathrm{subjects}}$",
            }
        )
).to_latex(
    latex_tables(ROOT) / "subject_count_per_year.tex",
    escape=False,
)

### Subject Count per Age Group

The following Code cell prepares a data frame called **project_results** that does not include subjects belonging to an age group that has less subjects than the first quantile of the *subject count per age group*.

In [ ]:
subject_count_per_age_group = (
    results
    .groupby("age_group")
    .size()
)

# _to_be_removed = subject_count_per_age_group[subject_count_per_age_group < int(np.round(subject_count_per_age_group.quantile(0.25)))]

# project_results = results[
#     ~results["age_group"].isin(_to_be_removed.index)
# ]

project_results = results.copy()

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = subject_count_per_age_group.to_frame("count")

(
    _df
        .rename_axis("Age Group", axis="index")
        .rename(
            columns={
                "count": r"$n_{\mathrm{subjects}}$",
            }
        )
).to_latex(
    latex_tables(ROOT) / "subject_count_per_ag.tex",
    escape=False,
)

### Results Grouped by…

#### Year – For the 1st Perspective

In [ ]:
_results = time_results.copy()

_ccdc_events = time_results[time_results["is_ccdc_event"] == True]

_group_keys = ["year"]
_cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "positive_rate",
    ),
    group_keys=tuple(_group_keys),
)

_results = (
    _results
    .groupby(_group_keys)[["full_name_of_repo", "is_ccdc_event"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)
_results = col_to_int(_results, INT_COLS_OF_SUMMARY)

_cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "n_distinct_channels",
    ),
    group_keys=tuple(_group_keys),
)
_ccdc_events = (
    _ccdc_events
    .groupby(_group_keys)[["full_name_of_repo", "detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)
_ccdc_events = col_to_int(_ccdc_events, INT_COLS_OF_SUMMARY)

_ccdc_events = _ccdc_events.rename(columns={"n_subjects": "n_ccdc_events"})
_ccdc_events = _ccdc_events.rename(columns={"n_repos": "n_pos_repos"})

_ccdc_events["n_ccdc_events_per_pos_repo"] = (
    _ccdc_events["n_ccdc_events"] / _ccdc_events["n_pos_repos"]
)

_ccdc_events["n_channels_per_pos_repo"] = (
    _ccdc_events["n_distinct_channels"] / _ccdc_events["n_pos_repos"]
)

_results = _results.join(
    _ccdc_events,
    how="left",
)

_results["n_ccdc_events_per_repo"] = (
    _results["n_ccdc_events"] / _results["n_repos"]
)

time_results_gb = _results.copy()

#### Age Group – For the 2nd Perspective

In [ ]:
_results = project_results.copy()

_ccdc_events = project_results[project_results["is_ccdc_event"] == True]

_group_keys = ["age_group"]
_cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "positive_rate",
    ),
    group_keys=tuple(_group_keys),
)

_results = (
    _results
    .groupby(_group_keys)[["full_name_of_repo", "is_ccdc_event"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)
_results = col_to_int(_results, INT_COLS_OF_SUMMARY)

_cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "n_distinct_channels",
    ),
    group_keys=tuple(_group_keys),
)
_ccdc_events = (
    _ccdc_events
    .groupby(_group_keys)[["full_name_of_repo", "detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)
_ccdc_events = col_to_int(_ccdc_events, INT_COLS_OF_SUMMARY)

_ccdc_events = _ccdc_events.rename(columns={"n_subjects": "n_ccdc_events"})
_ccdc_events = _ccdc_events.rename(columns={"n_repos": "n_pos_repos"})

_ccdc_events["n_ccdc_events_per_pos_repo"] = (
    _ccdc_events["n_ccdc_events"] / _ccdc_events["n_pos_repos"]
)

_ccdc_events["n_channels_per_pos_repo"] = (
    _ccdc_events["n_distinct_channels"] / _ccdc_events["n_pos_repos"]
)

_results = _results.join(
    _ccdc_events,
    how="left",
)

_results["n_ccdc_events_per_repo"] = (
    _results["n_ccdc_events"] / _results["n_repos"]
)

project_results_gb = _results.copy()

### Basic Demographics

What is the time range of the data?

In [ ]:
earliest_activity = repos["first_activity"].min()
most_recent_activity = repos["last_activity"].max()

print(f"Overall, the first (oldest) activity dates back to: {earliest_activity}.")
print(f"Again, overall, the last (most recent) activity dates back to: {most_recent_activity}.")

In [ ]:
years = pd.DataFrame({
    "year": range(
        earliest_activity.year,
        most_recent_activity.year + 1,
    )
})
years.set_index("year", inplace=True)

How many commits were processed?

In [ ]:
print(f"{results["commit_sha"].nunique()} commits were processed.")

How many subjects were processed and will be analyzed in this Jupyter Notebook?

In [ ]:
print(f"{len(results)} subjects were processed and will be analyzed here.")

Now, let's summarize the entire results data set:

In [ ]:
group_keys = []
cfg = SummarizeConfig(
    include_metrics=ALL_METRICS,
    group_keys=tuple(group_keys),
)

summary = summarize_subjects_configurable(results, cfg).to_frame().T
summary = col_to_int(summary, INT_COLS_OF_SUMMARY)
summary

### Path Demographics

Next, we group the subjects by path and create a summary for each group.

In [ ]:
_group_keys = ["path"]

_cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "n_commits",
    ),
    group_keys=tuple(_group_keys),
)

path_summaries = (
    results
        .groupby(_group_keys)[["full_name_of_repo", "commit_sha"]]
        .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)

path_summaries = col_to_int(path_summaries, INT_COLS_OF_SUMMARY)
path_summaries.sort_values("n_subjects", ascending=False, inplace=True)

In [ ]:
path_summaries

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = path_summaries

(
    _df
        .rename_axis("Path", axis="index")
        .rename(
            columns={
                "n_subjects": r"$n_{\mathrm{subjects}}$",
                "n_repos": r"$n_{\mathrm{repos}}$",
                "n_commits": r"$n_{\mathrm{commits}}$",
            }
        )
).to_latex(
    latex_tables(ROOT) / "path_summaries.tex",
    escape=False,
)

#### Shares of Distinct Path Values (e.g., README.txt)

##### CONTRIBUTING.md and README.txt

In [ ]:
results_gb_year_path = (
    time_results
    .groupby(["year", "path"])
    .size()
    .reset_index(name="count")
)

results_gb_year_path["share"] = (
    results_gb_year_path["count"]
    / results_gb_year_path.groupby("year")["count"].transform("sum")
)

In [ ]:
PATHS_OF_INTEREST = ["CONTRIBUTING.md", "README.txt"]

_df = (
    results_gb_year_path[
        results_gb_year_path["path"].isin(PATHS_OF_INTEREST)
    ]
    .copy()
)

_df = (
    _df
    .set_index(["year", "path"])
    .unstack(fill_value=0)
    .stack()
    .reset_index()
)

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="share",
    hue="path",
    marker="o",
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, _df["share"].max() * 1.1)
ax.margins(x=0.02)

ax.legend(title="Path", loc="best")

save_ieee_figure(fig, "path_shares_without_readme_md", save_pdf=True)

plt.show()

##### Including README.md

In [ ]:
# We make sure that README.md becomes a new color for the plot
order = ["CONTRIBUTING.md", "README.txt", "README.md"]

results_gb_year_path["path"] = pd.Categorical(
    results_gb_year_path["path"],
    categories=order,
    ordered=True
)

results_gb_year_path = results_gb_year_path.sort_values(["year", "path"])

In [ ]:
_df = results_gb_year_path.copy()

_df = (
    _df
    .set_index(["year", "path"])
    .unstack(fill_value=0)
    .stack()
    .reset_index()
)

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="share",
    hue="path",
    marker="o",
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, _df["share"].max() * 1.1)
ax.margins(x=0.02)

ax.legend(title="Path", loc="best")

save_ieee_figure(fig, "path_shares", save_pdf=True)

plt.show()

### Repository Demographics

Next, we take a look at the repository specific demographics…

#### Repository Count Overall

In [ ]:
repo_count = len(repos)

assert(results["full_name_of_repo"].nunique() == repo_count)
print(f"{repo_count} repositories contributed to the results.")

#### Repositories Come and Go…

Now, we create a table showing how many repositories were born in a year and how many repositories had their last activity in a year.

In [ ]:
_repos_copy = repos.copy()

_repos_copy["year_of_first_activity"] = _repos_copy["first_activity"].dt.year
_repos_copy["year_of_last_activity"] = _repos_copy["last_activity"].dt.year

_repos_come = (
    _repos_copy
    .groupby("year_of_first_activity")
    .size()
    .reset_index(name="count")
)

_repos_come.set_index("year_of_first_activity", drop=True, inplace=True)
_repos_come.rename_axis("year", inplace=True)

_repos_come = years.join(    
    _repos_come.rename_axis("year"),
    how="left"
)

_repos_come["count"] = _repos_come["count"].fillna(0).astype(int)
_repos_come = _repos_come.rename(columns={"count": "n_repos_come"})

_repos_go = (
    _repos_copy
    .groupby("year_of_last_activity")
    .size()
    .reset_index(name="count")
)

_repos_go.set_index("year_of_last_activity", drop=True, inplace=True)
_repos_go.rename_axis("year", inplace=True)

_repos_go = years.join(
    _repos_go.rename_axis("year"),
    how="left"
)

_repos_go["count"] = _repos_go["count"].fillna(0).astype(int)
_repos_go = _repos_go.rename(columns={"count": "n_repos_go"})

repos_come_and_go = _repos_come.join(
    _repos_go,
    how="left",
)

repos_come_and_go["n_dead"] = (
    repos_come_and_go["n_repos_go"]
    .cumsum()
    .shift(1, fill_value=0)
)

repos_come_and_go["n_alive"] = (
    repos_come_and_go["n_repos_come"].cumsum()
    - repos_come_and_go["n_dead"]
)

In [ ]:
repos_come_and_go

In summary, this approach ensures that:
- repositories contribute to the active count in the year they are created,
- and repositories that become inactive are only removed from the active population in the following year.

This results in a temporally consistent estimate of the number of active repositories over time.

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = repos_come_and_go[["n_repos_come", "n_repos_go", "n_alive"]]

(
    _df
        .rename_axis("Year", axis="index")
        .rename(
            columns={
                "n_repos_come": r"$n_{\mathrm{come}}$",
                "n_repos_go": r"$n_{\mathrm{go}}$",
                "n_alive": r"$n_{\mathrm{alive}}$",
            }
        )
).to_latex(
    latex_tables(ROOT) / "repos_come_and_go.tex",
    escape=False,
)

#### Age Groups Today

What do the age groups look like Today?

In [ ]:
group_keys = ["age_group"]

cfg = SummarizeConfig(
    include_metrics=(
        "n_repos",
    ),
    group_keys=tuple(group_keys),
)

repos_per_age_group_today = (
    repos
    .groupby(group_keys)[["full_name_of_repo"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

In [ ]:
repos_per_age_group_today

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = repos_per_age_group_today

(
    _df
        .rename_axis("Age Group", axis="index")
        .rename(
            columns={
                "n_repos": r"$n_{\mathrm{repos}}$",
            }
        )
).to_latex(
    latex_tables(ROOT) / "age_groups_today.tex",
    escape=False,
)

#### At What Age Do Repositories Have Their First Subject?

In [ ]:
_group_keys = ["age_group_at_first_subject"]

_cfg = SummarizeConfig(
    include_metrics=(
        "n_repos",
    ),
    group_keys=tuple(_group_keys),
)

age_at_first_subject = (
    repos
    .groupby(_group_keys)[["full_name_of_repo"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)

age_at_first_subject["share"] = (
    age_at_first_subject["n_repos"] / repo_count
)

In [ ]:
age_at_first_subject

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = age_at_first_subject

(
    _df
        .rename_axis("Age Group", axis="index")
        .rename(
            columns={
                "n_repos": r"$n_{\mathrm{repos}}$",
                "share": "",
            }
        )
).to_latex(
    latex_tables(ROOT) / "ag_at_first_subject.tex",
    escape=False,
    float_format="%.2f",
)

#### Survivors

In [ ]:
survivors = (
    repos
    .groupby("age_group")
    .size()
    .reset_index(name="count")
)

survivors.set_index("age_group", drop=True, inplace=True)

survivors["n_alive"] = survivors["count"][::-1].cumsum()[::-1]

survivors = survivors.rename(columns={"count": "n_dead"})

survivors["n_survivors"] = survivors["n_alive"] - survivors["n_dead"]

In [ ]:
survivors

LaTeX table export:

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = survivors[["n_alive", "n_survivors"]]

(
    _df
        .rename_axis("Age Group", axis="index")
        .rename(
            columns={
                "n_alive": r"$n_{\mathrm{alive}}$",
                "n_survivors": r"$n_{\mathrm{survivors}}$",
            }
        )
).to_latex(
    latex_tables(ROOT) / "age_group_survivors.tex",
    escape=False,
)

#### Per Year: Subject Count per Repository

In [ ]:
time_results_gb["n_subjects_per_active_repo"] = (
    time_results_gb["n_subjects"] / time_results_gb["n_repos"]
)

In [ ]:
_df = time_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="n_subjects_per_active_repo",
    marker="o",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Subject Count")

ax.set_ylim(0, _df["n_subjects_per_active_repo"].max() * 1.1)

ax.margins(x=0.02)

save_ieee_figure(fig, "subject_count_per_active_repo_per_year", save_pdf=True)

plt.show()

#### Each Year, How Many Repositories Have A Subject?

In [ ]:
time_results_gb = time_results_gb.join(
    repos_come_and_go["n_alive"],
    how="left",
)

In [ ]:
time_results_gb["share_active_repos"] = (
    time_results_gb["n_repos"] / time_results_gb["n_alive"]
)

In [ ]:
_df = time_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="share_active_repos",
    marker="o",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, _df["share_active_repos"].max() * 1.1)

ax.margins(x=0.02)

save_ieee_figure(fig, "active_repos_per_year", save_pdf=True)

plt.show()

#### Per Age Group: Subject Count Per Repository

Over the lifetime of a repo, how many subjects per active repo? How does that number evolve?

*-> How does the commitment RE project documentation evolve over the lifetime of a repository?*

In [ ]:
project_results_gb["n_subjects_per_active_repo"] = (
    project_results_gb["n_subjects"] / project_results_gb["n_repos"]
)

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="n_subjects_per_active_repo",
    marker="o",
    ax=ax
)

ax.set_xlabel("Age Group")
ax.set_ylabel("Subject Count")

ax.set_xticks(_df.index)

ax.margins(x=0.02)

save_ieee_figure(fig, "subject_count_per_active_repo_per_ag", save_pdf=True)

plt.show()

#### Per Age Group: Share of Active Repositories

In [ ]:
project_results_gb = project_results_gb.join(
    survivors["n_alive"],
    how="left",
)

In [ ]:
project_results_gb["share_active_repos"] = (
    project_results_gb["n_repos"] / project_results_gb["n_alive"]
)

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="share_active_repos",
    marker="o",
    ax=ax
)

ax.set_xlabel("Age Group")
ax.set_ylabel("")

ax.set_xticks(_df.index)

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, _df["share_active_repos"].max() * 1.1)
ax.margins(x=0.02)

save_ieee_figure(fig, "active_repos_per_ag", save_pdf=True)

plt.show()

#### Boxplot: Repository Age

In [ ]:
fig, ax = plt.subplots(figsize=ieee_figsize("single", ratio=1.2))

sns.boxplot(
    y=repos["age"],
    width=0.3,
    fliersize=2,
    linewidth=1,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 4,
    },
    ax=ax
)

ax.set_ylabel("Number of Days")
ax.set_xlabel("")
ax.set_xticks([])

ax.margins(x=0.1)

save_ieee_figure(fig, "repo_age_boxplot", save_pdf=True)

plt.show()

In [ ]:
# print(boxplot_stats(repos["age"]))

#### Boxplot: Subject Count Per Repository

In [ ]:
_df = (
    results
    .groupby(["full_name_of_repo"])
    .size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=ieee_figsize("single", ratio=1.2))

sns.boxplot(
    y=_df["count"],
    width=0.3,
    fliersize=2,
    linewidth=1,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 4,
    },
    ax=ax
)

ax.set_ylabel("Subject Count")
ax.set_xlabel("")
ax.set_xticks([])

ax.margins(x=0.1)

save_ieee_figure(fig, "subject_count_per_repo_boxplot", save_pdf=True)

plt.show()

In [ ]:
# print(boxplot_stats(results_gb_repo["count"]))

#### More Statistics…

Overall, how many CCDC events per repo (median)? How many channels per repo (median)?

In [ ]:
group_keys = ["full_name_of_repo"]

cfg = SummarizeConfig(
    include_metrics=(
        "positive_rate",
        "n_distinct_channels",
    ),
    group_keys=tuple(group_keys),
)

positive_rate_and_detected_channels_per_repo = (
    results
    .groupby(group_keys)[["is_ccdc_event", "detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

In [ ]:
positive_rate_and_detected_channels_per_repo.sort_values("n_distinct_channels", ascending=True, inplace=False).head(n=30)

In [ ]:
_foo = positive_rate_and_detected_channels_per_repo.copy()
print(len(_foo[_foo["n_distinct_channels"] == 0]))

In [ ]:
_group_keys = ["full_name_of_repo"]
_cfg = SummarizeConfig(
    include_metrics=(
        "n_distinct_channels",
    ),
    group_keys=tuple(_group_keys),
)
hm = (
    results
    .groupby(_group_keys)[["detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)
hm["n_distinct_channels"].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=ieee_figsize("single", ratio=1.2))

sns.boxplot(
    y=positive_rate_and_detected_channels_per_repo["positive_rate"],
    width=0.3,
    fliersize=2,
    linewidth=1,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 4,
    },
    ax=ax
)

ax.set_ylabel("")
ax.set_xlabel("")
ax.set_xticks([])

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, positive_rate_and_detected_channels_per_repo["positive_rate"].max() * 1.1)

ax.margins(x=0.1)

save_ieee_figure(fig, "ccdc_events_per_repo_boxplot", save_pdf=True)

plt.show()

In [ ]:
print(boxplot_stats(positive_rate_and_detected_channels_per_repo["positive_rate"]))

In [ ]:
fig, ax = plt.subplots(figsize=ieee_figsize("single", ratio=1.2))

sns.boxplot(
    y=positive_rate_and_detected_channels_per_repo["n_distinct_channels"],
    width=0.3,
    fliersize=2,
    linewidth=1,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 4,
    },
    ax=ax
)

ax.set_ylabel("Communication Channel Count")
ax.set_xlabel("")
ax.set_xticks([])

ax.margins(x=0.1)

save_ieee_figure(fig, "cc_count_per_repo_boxplot", save_pdf=True)

plt.show()

In [ ]:
print(boxplot_stats(positive_rate_and_detected_channels_per_repo["n_distinct_channels"]))

### CCDC Event Demographics

We filter our results to define a dedicated data frame that only includes CCDC events:

In [ ]:
ccdc_events = results[results["is_ccdc_event"] == True]

How many CCDC events are without any detected channel?

In [ ]:
empty_ccdc_events = ccdc_events[ccdc_events["detected_channels"] == ()]

print(f"There are {len(empty_ccdc_events)} CCDC events without any detected channel.")

_share = (
    len(empty_ccdc_events) / len(ccdc_events)
)

print(f"That is {_share*100:.2f}% of all CCDC events.")

When did these happen?

In [ ]:
_ccdc_events = time_results[time_results["is_ccdc_event"] == True].copy()

_ccdc_events["is_empty"] = _ccdc_events["detected_channels"] == ()

_ccdc_events["is_empty_label"] = _ccdc_events["is_empty"].map({
    True: "Empty Event",
    False: "Non-Empty Event"
})

_empty_vs_non_empty = (
    _ccdc_events
    .groupby(["year", "is_empty_label"])
    .size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=ieee_figsize("double"))

sns.barplot(
    data=_empty_vs_non_empty,
    x="year",
    y="count",
    hue="is_empty_label",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Count")

ax.margins(x=0.02)

ax.legend(title="CCDC Event Type", loc="upper right")

save_ieee_figure(fig, "empty_vs_non_empty_ccdc_events", save_pdf=True)

plt.show()

## 1st Perspective: Over Time (e.g., 2008–2025)

#### Share of CCDC Events per Year

##### Including 3-Year Rolling Mean

In [ ]:
_df = time_results_gb.copy()

_df["rolling_mean_3"] = (
    _df["positive_rate"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="positive_rate",
    marker="o",
    ax=ax,
)

sns.lineplot(
    data=_df,
    x="year",
    y="rolling_mean_3",
    marker="o",
    ax=ax,
    label="3-Year Rolling Mean"
)

ax.set_xlabel("")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.margins(x=0.02)

ax.legend(loc="best")

save_ieee_figure(fig, "share_of_ccdc_events_per_year_zoomed_in", save_pdf=True)

plt.show()

##### Normalized

In [ ]:
_df = time_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="positive_rate",
    marker="o",
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, 1)

ax.margins(x=0.02)

save_ieee_figure(fig, "share_of_ccdc_events_per_year_normalized", save_pdf=True)

plt.show()

#### Share of Positive Repositories per Year

In [ ]:
time_results_gb["share_pos_repos"] = (
    time_results_gb["n_pos_repos"] / time_results_gb["n_alive"]
)

In [ ]:
_df = time_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="share_active_repos",
    marker="o",
    ax=ax,
    label="Active Repository"
)

sns.lineplot(
    data=_df,
    x="year",
    y="share_pos_repos",
    marker="o",
    ax=ax,
    label="Positive Repository"
)

ax.set_xlabel("")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, _df["share_active_repos"].max() * 1.1)

ax.margins(x=0.02)

ax.legend(loc="best")

save_ieee_figure(fig, "share_of_pos_repos_per_year", save_pdf=True)

plt.show()

#### Mean CCDC Event Count per Active Repository per Year

In [ ]:
_df = time_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="n_subjects_per_active_repo",
    marker="o",
    ax=ax,
    label="Subject Count"
)

sns.lineplot(
    data=_df,
    x="year",
    y="n_ccdc_events_per_repo",
    marker="o",
    ax=ax,
    label="CCDC Event Count"
)

ax.set_xlabel("")
ax.set_ylabel("Subject Count")

ax.set_ylim(0, _df["n_subjects_per_active_repo"].max() * 1.1)

ax.margins(x=0.02)

ax.legend(loc="best")

save_ieee_figure(fig, "mean_ccdc_event_count_per_repo_per_year", save_pdf=True)

plt.show()

#### Channel Diversity

In [ ]:
_df = time_results_gb.copy()

_df["rolling_mean_5"] = (
    _df["n_distinct_channels"].rolling(5, center=True).mean()
)

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="n_distinct_channels",
    marker="o",
    ax=ax,
)

sns.lineplot(
    data=_df,
    x="year",
    y="rolling_mean_5",
    marker="o",
    ax=ax,
    label="5-year Rolling Mean"
)

ax.set_xlabel("")
ax.set_ylabel("Communication Channel Count")

ax.set_ylim(0, 36)

ax.legend(loc="best")

ax.margins(x=0.02)

save_ieee_figure(fig, "cc_diversity_per_year", save_pdf=True)

plt.show()

#### Heatmap

We introduce a matrix that has a column for each channel and lists the channel count for a channel for a year.

In [ ]:
cc_count_matrix = (
    results
    .explode("detected_channels")
    .groupby(["year", "detected_channels"])
    .size()
    .unstack(fill_value=0)
)

cc_count_matrix.columns.name = None
cc_count_matrix = cc_count_matrix.loc[:, cc_count_matrix.iloc[-1].sort_values(ascending=False).index]
# cc_count_matrix = cc_count_matrix[cc_count_matrix.sum().sort_values(ascending=False).index]

In [ ]:
cc_count_matrix

Now, we plot a heatmap for the channel count matrix.

In [ ]:
fig, ax = plt.subplots(figsize=ieee_figsize("double"))

im = ax.imshow(cc_count_matrix, aspect="auto")

ax.set_xlabel("Communication Channel")
ax.set_ylabel("")

ax.set_xticks(range(len(cc_count_matrix.columns)))
ax.set_xticklabels(cc_count_matrix.columns, rotation=90)

ax.set_yticks(range(len(cc_count_matrix.index)))
ax.set_yticklabels(cc_count_matrix.index)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("CCDC Event Count")

plt.tight_layout()

save_ieee_figure(fig, "cc_heatmap_per_year", save_pdf=True)

plt.show()

#### Channels per CCDC Event

In [ ]:
_ccdc_events = time_results[time_results["is_ccdc_event"] == True]

_vc_detected_channels_per_year = (
    _ccdc_events
    .groupby("year")[["detected_channels"]]
    .apply(vc_detected_channels)
)

_vc_stats_per_year = (
    _vc_detected_channels_per_year
    .reset_index()
    .groupby("year")["count"]
    .apply(value_counts_stats)
).unstack(fill_value=0)
_vc_stats_per_year = col_to_int(_vc_stats_per_year, INT_COLS_OF_VC_STATS)

vc_detected_channels_per_year = _vc_detected_channels_per_year.reset_index()

cc_count_per_year = _vc_stats_per_year[["total_count"]].join(
    time_results_gb["n_ccdc_events"],
    how="left",
)

cc_count_per_year["ratio"] = (
    cc_count_per_year["total_count"] / cc_count_per_year["n_ccdc_events"]
)

In [ ]:
_r = time_results.copy()

_c = _r[_r["is_ccdc_event"] == True].copy()

_c["is_empty"] = _c["detected_channels"] == ()

_empty = _c[_c["is_empty"] == True].copy()
_non_empty = _c[_c["is_empty"] == False].copy()

print(f"From 2013 to 2025, {len(_c)} CCDC events were created.")
print(f"From 2013 to 2025, {len(_non_empty)} non-empty CCDC events were created.")
print(f"That is {(len(_non_empty) / len(_c))*100:.2f}% of all CCDC events.")

In [ ]:
_exploded = (
    _non_empty
    .explode("detected_channels")
    .groupby(["year"])
    .size()
    .reset_index(name="dc_count")
)

_counts = (
    _non_empty
    .groupby(["year"])
    .size()
    .reset_index(name="ne_event_count")
)

_total_counts = (
    _c
    .groupby(["year"])
    .size()
    .reset_index(name="event_count")
)

_exploded = _exploded.join(_total_counts["event_count"])
_exploded = _exploded.join(_counts["ne_event_count"])

_exploded["mean_ccc_per_event"] = (
    _exploded["dc_count"] / _exploded["event_count"]
)
_exploded["mean_ccc_per_ne_event"] = (
    _exploded["dc_count"] / _exploded["ne_event_count"]
)

_df = _exploded.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="year",
    y="mean_ccc_per_event",
    marker="o",
    ax=ax,
    label="Including Empty CCDC Events"
)

sns.lineplot(
    data=_df,
    x="year",
    y="mean_ccc_per_ne_event",
    marker="o",
    ax=ax,
    label="Excluding Empty CCDC Events"
)

ax.set_xlabel("")
ax.set_ylabel("Communication Channel Count")
ax.set_ylim(0, 4.6)
ax.legend(loc="best")
ax.margins(x=0.02)

save_ieee_figure(fig, "mean_cc_count_per_ccdc_event_per_year", save_pdf=True)

plt.show()

#### Channel Counts

In [ ]:
cc_counts_for_time_results = (
    time_results["detected_channels"]
        .explode()
        .dropna()
        .value_counts()
        .rename_axis("channel")
        .reset_index(name="count")
)

#### Top *n* Most Popular Channels

In [ ]:
n = 5

top_5_per_year = (
    vc_detected_channels_per_year
        .sort_values(["year", "count"], ascending=[True, False])
        .groupby("year", dropna=False, sort=False)
        .head(n)
        .reset_index(drop=True)
)

n = 10

top_10_per_year = (
    vc_detected_channels_per_year
        .sort_values(["year", "count"], ascending=[True, False])
        .groupby("year", dropna=False, sort=False)
        .head(n)
        .reset_index(drop=True)
)

How many channels have ever made it into the top 10 of year?

In [ ]:
print(f"Answer: {top_5_per_year["channel"].nunique()}")

print(f"These are:\n{top_5_per_year["channel"].unique()}")

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = cc_counts_for_time_results[
    cc_counts_for_time_results["channel"].isin(top_5_per_year["channel"].unique())
]

_df = (
    _df
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "cc_annual_top_5.tex",
    escape=False,
    index=True,
)

In [ ]:
top_5_per_year[top_5_per_year["year"] == 2016]

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = top_10_per_year[top_10_per_year["year"] == 2016].copy()

_df = (
    _df
    .drop(columns="year")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_10_channels_2016.tex",
    escape=False,
    index=True,
)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = top_5_per_year[top_5_per_year["year"] == 2013].copy()

_df = (
    _df
    .drop(columns="year")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_5_channels_2013.tex",
    escape=False,
    index=True,
)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = top_5_per_year[top_5_per_year["year"] == 2019].copy()

_df = (
    _df
    .drop(columns="year")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_5_channels_2019.tex",
    escape=False,
    index=True,
)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = top_5_per_year[top_5_per_year["year"] == 2022].copy()

_df = (
    _df
    .drop(columns="year")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_5_channels_2022.tex",
    escape=False,
    index=True,
)

## 2nd Perspective: The Repository Life Cycle

#### Share of CCDC Events per Age Group – **not** grouped by repository

In [ ]:
share_of_ccdc_events_per_ag = project_results_gb.copy()

share_of_ccdc_events_per_ag["rolling_mean_3"] = (
    share_of_ccdc_events_per_ag["positive_rate"].rolling(3, center=True).mean()
)

share_of_ccdc_events_per_ag["rolling_mean_5"] = (
    share_of_ccdc_events_per_ag["positive_rate"].rolling(5, center=True).mean()
)

##### Including 5-Year Rolling Mean

In [ ]:
_df = share_of_ccdc_events_per_ag.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="positive_rate",
    marker="o",
    ax=ax,
)

sns.lineplot(
    data=_df,
    x="age_group",
    y="rolling_mean_5",
    marker="o",
    ax=ax,
    label="5-year Rolling Mean"
)

ax.set_xlabel("Age Group")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.legend(loc="best")

ax.margins(x=0.02)

save_ieee_figure(fig, "share_of_ccdc_events_per_ag_zoomed_in", save_pdf=True)

plt.show()

##### Normalized

In [ ]:
_df = share_of_ccdc_events_per_ag.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="positive_rate",
    marker="o",
    ax=ax,
)

ax.set_xlabel("Age Group")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 1)

ax.margins(x=0.02)

save_ieee_figure(fig, "share_of_ccdc_events_per_ag_normalized", save_pdf=True)

plt.show()

#### Share of Positive Repositories per Age Group

This is the likelihood of a repository having a CCDC events assuming/if the repository is still alive meaning it will be updated in the future still.

In [ ]:
project_results_gb["share_pos_repos"] = (
    project_results_gb["n_pos_repos"] / project_results_gb["n_alive"]
)

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="share_active_repos",
    marker="o",
    ax=ax,
    label="Active Repository"
)

sns.lineplot(
    data=_df,
    x="age_group",
    y="share_pos_repos",
    marker="o",
    ax=ax,
    label="Positive Repository"
)

ax.set_xlabel("Age Group")
ax.set_ylabel("")

from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

ax.set_ylim(0, _df["share_active_repos"].max() * 1.1)

ax.margins(x=0.02)

ax.legend(loc="best")

save_ieee_figure(fig, "share_of_pos_repos_per_ag", save_pdf=True)

plt.show()

#### Mean CCDC Event Count per Positive Repository per Age Group

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="n_subjects_per_active_repo",
    marker="o",
    ax=ax,
    label="Subject Count"
)

sns.lineplot(
    data=_df,
    x="age_group",
    y="n_ccdc_events_per_repo",
    marker="o",
    ax=ax,
    label="CCDC Event Count"
)

ax.set_xlabel("Age Group")
ax.set_ylabel("Subject Count")

ax.set_ylim(0, _df["n_subjects_per_active_repo"].max() * 1.1)

ax.margins(x=0.02)

ax.legend(loc="best")

save_ieee_figure(fig, "mean_ccdc_event_count_per_repo_per_ag", save_pdf=True)

plt.show()

#### Channel Diversity

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="n_distinct_channels",
    marker="o",
    ax=ax,
)

ax.set_xlabel("Age Group")
ax.set_ylabel("Communication Channel Count")

ax.set_ylim(0, 36)

ax.margins(x=0.02)

save_ieee_figure(fig, "cc_diversity_per_ag", save_pdf=True)

plt.show()

#### Heatmap

In [ ]:
cc_matrix_gb_age_group = (
    results
    .explode("detected_channels")
    .groupby(["age_group", "detected_channels"])
    .size()
    .unstack(fill_value=0)
)

cc_matrix_gb_age_group.columns.name = None
cc_matrix_gb_age_group = cc_matrix_gb_age_group.loc[:, cc_matrix_gb_age_group.iloc[0].sort_values(ascending=False).index]
# cc_matrix_gb_age_group = cc_matrix_gb_age_group[cc_matrix_gb_age_group.sum().sort_values(ascending=False).index]

_df = cc_matrix_gb_age_group.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("double"))

im = ax.imshow(_df, aspect="auto")

ax.set_xlabel("Communication Channel")
ax.set_ylabel("Age Group")

ax.set_xticks(range(len(_df.columns)))
ax.set_xticklabels(_df.columns, rotation=90)

ax.set_yticks(range(len(_df.index)))
ax.set_yticklabels(_df.index)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("CCDC Event Count")

plt.tight_layout()

save_ieee_figure(fig, "cc_heatmap_per_ag", save_pdf=True)

plt.show()

#### Channels per CCDC Event

In [ ]:
_ccdc_events = project_results[project_results["is_ccdc_event"] == True]

_vc_detected_channels_per_age_group = (
    _ccdc_events
    .groupby("age_group")[["detected_channels"]]
    .apply(vc_detected_channels)
)

_vc_stats_per_age_group = (
    _vc_detected_channels_per_age_group
    .reset_index()
    .groupby("age_group")["count"]
    .apply(value_counts_stats)
).unstack(fill_value=0)
_vc_stats_per_age_group = col_to_int(_vc_stats_per_age_group, INT_COLS_OF_VC_STATS)

vc_detected_channels_per_age_group = _vc_detected_channels_per_age_group.reset_index()

cc_count_per_age_group = _vc_stats_per_age_group[["total_count"]].join(
    project_results_gb["n_ccdc_events"],
    how="left",
)

cc_count_per_age_group["ratio"] = (
    cc_count_per_age_group["total_count"] / cc_count_per_age_group["n_ccdc_events"]
)

In [ ]:
cc_count_per_age_group

In [ ]:
_df = cc_count_per_age_group.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="ratio",
    marker="o",
    ax=ax,
)

ax.set_xlabel("Age Group")
ax.set_ylabel("Communication Channel Count")

ax.set_ylim(0, 4.6)

ax.margins(x=0.02)

save_ieee_figure(fig, "mean_cc_count_per_ccdc_event_per_ag", save_pdf=True)

plt.show()

In [ ]:
_r = project_results.copy()

_c = _r[_r["is_ccdc_event"] == True].copy()

_c["is_empty"] = _c["detected_channels"] == ()

_empty = _c[_c["is_empty"] == True].copy()
_non_empty = _c[_c["is_empty"] == False].copy()

print(f"Across all age groups, {len(_c)} CCDC events were created.")
print(f"Across all age groups, {len(_non_empty)} non-empty CCDC events were created.")
print(f"That is {(len(_non_empty) / len(_c))*100:.2f}% of all CCDC events.")

In [ ]:
_exploded = (
    _non_empty
    .explode("detected_channels")
    .groupby(["age_group"])
    .size()
    .reset_index(name="dc_count")
)

_counts = (
    _non_empty
    .groupby(["age_group"])
    .size()
    .reset_index(name="ne_event_count")
)

_total_counts = (
    _c
    .groupby(["age_group"])
    .size()
    .reset_index(name="event_count")
)

_exploded = _exploded.join(_total_counts["event_count"])
_exploded = _exploded.join(_counts["ne_event_count"])

_exploded["mean_ccc_per_event"] = (
    _exploded["dc_count"] / _exploded["event_count"]
)
_exploded["mean_ccc_per_ne_event"] = (
    _exploded["dc_count"] / _exploded["ne_event_count"]
)

_df = _exploded.copy()

fig, ax = plt.subplots(figsize=ieee_figsize("single"))

sns.lineplot(
    data=_df,
    x="age_group",
    y="mean_ccc_per_event",
    marker="o",
    ax=ax,
    label="Including Empty CCDC Events"
)

sns.lineplot(
    data=_df,
    x="age_group",
    y="mean_ccc_per_ne_event",
    marker="o",
    ax=ax,
    label="Excluding Empty CCDC Events"
)

ax.set_xlabel("Age Group")
ax.set_ylabel("Communication Channel Count")
ax.set_ylim(0, 4.6)
ax.legend(loc="best")
ax.margins(x=0.02)

save_ieee_figure(fig, "mean_cc_count_per_ccdc_event_per_ag", save_pdf=True)

plt.show()

#### Top 5 Channels

In [ ]:
_top_5_per_age_group = (
    vc_detected_channels_per_age_group
        .sort_values(["age_group", "count"], ascending=[True, False])
        .groupby("age_group", dropna=False, sort=False)
        .head(5)
        .reset_index(drop=True)
)

_top_10_per_age_group = (
    vc_detected_channels_per_age_group
        .sort_values(["age_group", "count"], ascending=[True, False])
        .groupby("age_group", dropna=False, sort=False)
        .head(10)
        .reset_index(drop=True)
)

How many channels have ever made it into the top 5 of an age group?

In [ ]:
print(f"Answer: {_top_5_per_age_group["channel"].nunique()}")

# print(f"These are:\n{_top_5_per_age_group["channel"].unique()}")

##### First Year (Age Group 0)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = _top_10_per_age_group[_top_10_per_age_group["age_group"] == 0].copy()

_df = (
    _df
    .drop(columns="age_group")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_10_channels_ag_0.tex",
    escape=False,
    index=True,
)

vs. Age Group 9

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = _top_10_per_age_group[_top_10_per_age_group["age_group"] == 9].copy()

_df = (
    _df
    .drop(columns="age_group")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_10_channels_ag_9.tex",
    escape=False,
    index=True,
)

##### 4th Year (Age Group 3)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = _top_5_per_age_group[_top_5_per_age_group["age_group"] == 3].copy()

_df = (
    _df
    .drop(columns="age_group")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_5_channels_ag_3.tex",
    escape=False,
    index=True,
)

##### 7th Year (Age Group 8)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = _top_5_per_age_group[_top_5_per_age_group["age_group"] == 6].copy()

_df = (
    _df
    .drop(columns="age_group")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_5_channels_ag_6.tex",
    escape=False,
    index=True,
)

##### 10th Year (Age Group 9)

In [ ]:
latex_tables(ROOT).mkdir(parents=True, exist_ok=True)

_df = _top_5_per_age_group[_top_5_per_age_group["age_group"] == 9].copy()

_df = (
    _df
    .drop(columns="age_group")
    .reset_index(drop=True)
    .rename(
        columns={
            "channel": "Channel",
            "count": "Count",
        }
    )
)

_df.index = _df.index + 1
_df.index.name = "Rank"

_df.to_latex(
    latex_tables(ROOT) / "top_5_channels_ag_9.tex",
    escape=False,
    index=True,
)